In [1]:
# ── experiment_runner.py ──────────────────────────────────────────────────
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
import torch
import numpy as np
from tqdm import tqdm
import pandas as pd

from src.dataloading.epiconfig import EpiConfig
from src.dataloading.epidataorchestration import EpiDataOrchestrator
from src.dataloading.dataloaders import (
    BaseLineDataLoaderManager, DeepDataLoaderManager, GraphDataLoaderManager
)
from src.models import (
    PersistenceModel, ClimateologyModel, ClimaScaleModel,
    LSTMModel, GCN2Model
)
from src.evaluation import Evaluator


# ── config ────────────────────────────────────────────────────────────────

BASE_CONFIG = dict(
    disease                 = 'influenza',
    country                 = 'germany',
    level                   = 'nuts3',
    min_date                = '2012-06-01',
    max_date                = '2020-06-01',
    feature_popsize         = True,
    feature_popdens         = True,
    feature_gisd            = False,
    feature_popage          = False,
    feature_kreise_classes  = False,
    feature_borders         = False,
    sequence_length         = 4,
    normalization_method    = 'zscore',
    log_transform           = ['incidence'],
)

GLOBAL_HPARAMS = dict(
    lr                  = 0.0001,
    n_epochs            = 250,
    scheduler           = 'plateau',
    scheduler_kwargs    = {'mode': 'min', 'factor': 0.2, 'patience': 5},
    min_delta           = 0.0001,
    loss                = 'mse',
    patience            = 20,
)

MODEL_HPARAMS = dict(
    gcn  = dict(dropout=0.3, self_loops=False),
    lstm = dict(),   # uses defaults: hidden_size=64, num_layers=1, dropout=0.2
)

HORIZON_LEADTIMES   = list(range(1, 11))   # hl1 to hl10
SEEDS               = [42, 123, 456, 8, 21]        # 3 seeds for variance estimation
GRAPH_KEYS          = {
    'identity'  : 'graph1',
    'geo'       : 'graph2',
    'random'    : 'graph3',
    'commuter'  : 'graph16',
}


# ── result container ──────────────────────────────────────────────────────

@dataclass
class HorizonResult:
    horizon:    int
    seed:       int
    evaluator:  Evaluator
    model_names: List[str]


# ── runner ────────────────────────────────────────────────────────────────

def run_experiment(
    horizon_leadtimes:  List[int]       = HORIZON_LEADTIMES,
    seeds:              List[int]       = SEEDS,
    base_config:        dict            = BASE_CONFIG,
    global_hparams:     dict            = GLOBAL_HPARAMS,
    model_hparams:      dict            = MODEL_HPARAMS,
    graph_keys:         dict            = GRAPH_KEYS,
    verbose:            int             = -1,
) -> Dict[int, Dict[int, HorizonResult]]:
    """
    Run the full influenza experiment across horizons and seeds.

    Returns
    -------
    results[horizon][seed] -> HorizonResult
    """

    EXP_dir = 'experiment_1'

    # ── build orchestrators once per horizon (expensive) ─────────────────
    dataorchs = {}
    for hl in tqdm(horizon_leadtimes, desc='orchestrators'):
        cfg = EpiConfig(horizon_leadtime=hl, **base_config)
        dataorchs[hl] = EpiDataOrchestrator(cfg).build()

    results: Dict[int, Dict[int, HorizonResult]] = {
        hl: {} for hl in horizon_leadtimes
    }

    # ── outer loop: horizons ──────────────────────────────────────────────
    for hl in horizon_leadtimes:
        data_orch = dataorchs[hl]
        suffix    = f'_hl{hl}'

        # baselines don't depend on seed — run once
        baseline_dlm = BaseLineDataLoaderManager(data_orch)
        persistence  = PersistenceModel(baseline_dlm, f'persistence{suffix}')
        climatology  = ClimateologyModel(baseline_dlm, f'climateology{suffix}')
        climascale   = ClimaScaleModel(baseline_dlm, f'climascale{suffix}')

        persistence.forecast('test')
        climatology.forecast('test')
        climascale.forecast('test')

        baselines = [persistence, climatology, climascale]

        # ── inner loop: seeds ─────────────────────────────────────────────
        for seed in seeds:
            print(f"\n{'='*50}")
            print(f"  hl={hl}  seed={seed}")
            print(f"{'='*50}")

            torch.manual_seed(seed)
            np.random.seed(seed)

            seed_suffix = f'{suffix}_s{seed}'

            # dataloaders
            deep_dlm     = DeepDataLoaderManager(data_orch).build()
            graph_dlms   = {
                name: GraphDataLoaderManager(data_orch)
                          .retrieve_static_graph(key)
                          .build()
                for name, key in graph_keys.items()
            }

            # LSTM
            lstm = LSTMModel(deep_dlm, name=f'lstm{seed_suffix}', verbose=verbose)
            lstm.set_model_hparams(**model_hparams['lstm'])
            lstm.set_global_hparams(**global_hparams)
            lstm.train()
            lstm.forecast('test')
            lstm.save_model(dir = EXP_dir)

            # GCNs
            gcns = {}
            for graph_name, dlm in graph_dlms.items():
                gcn = GCN2Model(dlm, name=f'gcn_{graph_name}{seed_suffix}', verbose=verbose)
                gcn.set_model_hparams(**model_hparams['gcn'])
                gcn.set_global_hparams(**global_hparams)
                gcn.train()
                gcn.forecast('test')
                gcns[graph_name] = gcn
                gcn.save_model(dir = EXP_dir)

            
            # evaluate
            all_models = baselines + [lstm] + list(gcns.values())
            evaluator  = Evaluator(all_models)
            evaluator.add_evaluation(horizon=0, dataset='test')

            results[hl][seed] = HorizonResult(
                horizon     = hl,
                seed        = seed,
                evaluator   = evaluator,
                model_names = [m.name for m in all_models],
            )

            print(f"  hl={hl} seed={seed} done")

    return results, dataorchs


# ── analysis ──────────────────────────────────────────────────────────────

def summarise_results(
    results:            Dict[int, Dict[int, HorizonResult]],
    horizon_leadtimes:  List[int] = HORIZON_LEADTIMES,
    seeds:              List[int] = SEEDS,
    metric:             str       = 'ccc',
) -> pd.DataFrame:
    """
    Aggregate metric across seeds per model per horizon.
    Returns a DataFrame with mean ± std over seeds.
    """
    records = []

    for hl in horizon_leadtimes:
        # persistence CCC for delta computation
        pers_values = []

        for seed in seeds:
            if seed not in results[hl]:
                continue
            ev = results[hl][seed].evaluator
            metrics = ev.data_compilation.get_data(0, 'test')['metrics'].copy()
            metrics['model'] = metrics['model'].astype(str)

            pers_name = f'persistence_hl{hl}'
            pers_row  = metrics[metrics['model'] == pers_name]
            if not pers_row.empty:
                pers_values.append(pers_row[metric].mean())

        pers_mean = np.mean(pers_values) if pers_values else np.nan

        for seed in seeds:
            if seed not in results[hl]:
                continue
            ev = results[hl][seed].evaluator
            metrics = ev.data_compilation.get_data(0, 'test')['metrics'].copy()
            metrics['model'] = metrics['model'].astype(str)

            for model_name in metrics['model'].unique():
                # strip seed suffix for clean label
                label = model_name.replace(f'_s{seed}', '').replace(f'_hl{hl}', '')
                val   = metrics[metrics['model'] == model_name][metric].mean()
                records.append({
                    'horizon':      hl,
                    'seed':         seed,
                    'model':        label,
                    metric:         val,
                    f'{metric}_delta': val - pers_mean,
                })

    df = pd.DataFrame(records)

    # aggregate over seeds
    summary = (
        df.groupby(['horizon', 'model'])
        .agg(
            mean      = (metric,              'mean'),
            std       = (metric,              'std'),
            mean_delta= (f'{metric}_delta',   'mean'),
            std_delta = (f'{metric}_delta',   'std'),
        )
        .round(4)
        .reset_index()
    )

    return df, summary


def print_summary_table(summary: pd.DataFrame, metric: str = 'ccc'):
    """Print mean delta over persistence per model per horizon."""
    pivot = summary.pivot_table(
        index   = 'model',
        columns = 'horizon',
        values  = 'mean_delta'
    ).round(4)

    print(f"\n── Mean {metric} delta over persistence (mean across seeds) ──")
    print(pivot.to_string())

    print(f"\n── Std across seeds (model × horizon) ────────────────────────")
    pivot_std = summary.pivot_table(
        index   = 'model',
        columns = 'horizon',
        values  = 'std_delta'
    ).round(4)
    print(pivot_std.to_string())

In [2]:
results, dataorchs = run_experiment(
    horizon_leadtimes = list(range(1, 8)),
    seeds             = [41, 123, 456],
    verbose         = -2
)

orchestrators: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:16<00:00,  2.42s/it]



  hl=1  seed=41
✓ Model saved: Wissdaten/ZKI-PH4/deschrijvers_wissdaten/project_utilities/infectious_disease_gnn/models/experiment_1/lstm_hl1_s41.pt
✓ Model saved: Wissdaten/ZKI-PH4/deschrijvers_wissdaten/project_utilities/infectious_disease_gnn/models/experiment_1/gcn_identity_hl1_s41.pt
✓ Model saved: Wissdaten/ZKI-PH4/deschrijvers_wissdaten/project_utilities/infectious_disease_gnn/models/experiment_1/gcn_geo_hl1_s41.pt


KeyboardInterrupt: 

In [ ]:
results

{1: {42: HorizonResult(horizon=1, seed=42, evaluator=<Evaluator(models: ['persistence_hl1', 'climateology_hl1', 'climascale_hl1', 'lstm_hl1_s42', 'gcn_identity_hl1_s42', 'gcn_geo_hl1_s42', 'gcn_random_hl1_s42', 'gcn_commuter_hl1_s42'],
  	data_compilation: <EvaluationPredictionsCompilation({'test': ['horizon_0']})>
  	metric_calculator: <PointRegressionMetricsCalculator(supported_metrics: ['bcf', 'ccc', 'mae', 'mape', 'mbe', 'mse', 'pearson', 'r2', 'rmse', 'smape', 'spearman', 'vr'])>)>, model_names=['persistence_hl1', 'climateology_hl1', 'climascale_hl1', 'lstm_hl1_s42', 'gcn_identity_hl1_s42', 'gcn_geo_hl1_s42', 'gcn_random_hl1_s42', 'gcn_commuter_hl1_s42']),
  123: HorizonResult(horizon=1, seed=123, evaluator=<Evaluator(models: ['persistence_hl1', 'climateology_hl1', 'climascale_hl1', 'lstm_hl1_s123', 'gcn_identity_hl1_s123', 'gcn_geo_hl1_s123', 'gcn_random_hl1_s123', 'gcn_commuter_hl1_s123'],
  	data_compilation: <EvaluationPredictionsCompilation({'test': ['horizon_0']})>
  	metric